# Fleet Management Data Collection

If you've run through the road following, your should be familiar following three steps

1.  Data collection
2.  Training
3.  Deployment

In this notebook, we'll do the same exact thing!  Except, instead of classification, you'll learn a different fundamental technique, **object detection**, that we'll use to
enable JetBot to follow an object (or really, any object such as jetbot).

1. Place the JetBot in different positions on a path (offset from center, different angles, etc)

>  Remember from road following, data variation is key!

2. Display the live camera feed from the robot
3. Using a gamepad controller, place a 'green dot', which corresponds to the left top point and right top point that can cover the boundary Jetbot, on the image.
4. Store as the label value, the left top point(x1, y1) and right top point (x2, y2) values, of this green dot along with the image from the Jetbot's camera, which is the bounding box value of the Jetbot

Then, in the training notebook, we'll train a neural network to predict the label values of our label.  In the live demo, we'll use
the predicted bounding value (x1, y1) and (x2, y2) to compute an approximate size value and position od Jetbot.

So how do you decide exactly where to place the target for this example?  Here is a guide we think may help

1.  Look at the live video feed from the camera
2.  Place a target Jetbot where it could stay.

Assuming our deep learning model works as intended, these labeling guidelines should ensure the robot can be accurately detected

### Import Libraries

So lets get started by importing all the required libraries for "data collection" purpose. We will mainly use OpenCV to visualize and save image with labels. Libraries such as uuid, datetime are used for image naming. 

In [ ]:
# IPython Libraries for display and widgets
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# Camera and Motor Interface for JetBot
from jetbot import Robot, Camera, bgr8_to_jpeg

# Basic Python packages for image annotation
from uuid import uuid1
import os
import json
import glob
import datetime
import numpy as np
import cv2
import time

### Display Live Camera Feed

First, let's initialize and display our camera like we did in the teleoperation notebook. 

We use Camera Class from JetBot to enable CSI MIPI camera. Our neural network takes a 224x224 pixel image as input. We'll set our camera to that size to minimize the filesize of our dataset (we've tested that it works for this task). In some scenarios it may be better to collect data in a larger image size and downscale to the desired size later.

In [ ]:
# camera = Camera()
from jetbot import DataCollection
dc = DataCollection()

cam_width = dc.camera.width
cam_height = dc.camera.height
cam_width_display = dc.camera.width_display
cam_height_display = dc.camera.height_display

widget_width = 360
widget_height = 360

image_widget = widgets.Image(format='jpeg', width=widget_width, height=widget_height)
target_widget = widgets.Image(format='jpeg', width=widget_width, height=widget_height)

x_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='x')
y_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='y')

time.sleep(1)

def update_widget_image(change):
    dc.get_widget_image()

def update_xy_image(change):
    dc.get_xy_image(x_slider.value, y_slider.value)

dc.camera.observe(update_widget_image, 'value')
dc.camera.observe(update_xy_image, 'value')
traitlets.dlink((dc, 'widget_image'), (image_widget, 'value'))
traitlets.dlink((dc, 'xy_image'), (target_widget, 'value'))         # display image with xy coordinates data

display(widgets.HBox([image_widget, target_widget]), x_slider, y_slider)


# display buttons for start and stop running
button_layout = widgets.Layout(width='150px', height='40px', align_self='center')
redo_button = widgets.Button(description='Stop', tooltip='Click to stop running', icon='stop', layout=button_layout)
redo_button.style.button_color='Red'
save_button = widgets.Button(description='Start', tooltip='Click to start running', icon='play', layout=button_layout)
save_button.style.button_color='lightBlue'

button_box = widgets.HBox([save_button, redo_button], layout=widgets.Layout(justify_content='space-around', width='30%'))
save_button.disabled=True
redo_button.disabled=True

count_xy = 2
bbox = []


### Create Gamepad Controller

This step is similar to "Teleoperation" task. In this task, we will use gamepad controller to label images.

The first thing we want to do is create an instance of the Controller widget, which we'll use to label images with "x" and "y" values as mentioned in introduction. The Controller widget takes a index parameter, which specifies the number of the controller. This is useful in case you have multiple controllers attached, or some gamepads appear as multiple controllers. To determine the index of the controller you're using,

Visit http://html5gamepad.com.
Press buttons on the gamepad you're using
Remember the index of the gamepad that is responding to the button presses
Next, we'll create and display our controller using that index.

In [ ]:
controller = widgets.Controller(index=0)

display(controller)

### Connect Gamepad Controller to Label Images

Now, even though we've connected our gamepad, we haven't yet attached the controller to label images! We'll connect that to the left and right vertical axes using the dlink function. The dlink function, unlike the link function, allows us to attach a transform between the source and target. 

### Collect data

The following block of code will display the live image feed, as well as the number of images we've saved.  We store
the target X, Y values by

1. Place the green dot on the target. It would be better around the central line of image window
2. Press 'down' on the DPAD to save

This will store a file in the ``dataset_xy`` folder with files named

``xy_<x value>_<y value>_<uuid>.jpg``

where `<x value>` and `<y value>` are the coordinates **in pixel (not in percentage)** (count from the top left corner).

When we train, we load the images and parse the x, y values from the filename

In [ ]:
DATASET_DIR = 'jetbot_bbox'

# we have this "try/except" statement because these next functions can throw an error if the directories exist already
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Directories not created because they already exist')

In [ ]:
for b in controller.buttons:
    b.unobserve_all()

In [ ]:
count_widget = widgets.IntText(description='count', value=len(glob.glob(os.path.join(DATASET_DIR, '*.jpg'))))

# def xy_uuid(x, y):
#    return 'xy_%03d_%03d_%s' % (x * cam_width / 2 + cam_width / 2, y * cam_height / 2 + cam_height / 2, uuid1())
snap_key = 5
save_key = 6
redo_key = 7

def snapshot_bbox(change):
    global count_xy, bbox, snap_key, save_key, redo_key
    if count_xy==0:
        controller.buttons[snap_key].unobserve()
        controller.buttons[save_key].observe(save_bbox, names='value')
        controller.buttons[redo_key].observe(redo, names='value')
        return

    dc.get_xy_point_image(x_slider.value, y_slider.value)
    count_xy-=1
    bbox.append((x_slider.value, y_slider.value))

    if count_xy==0:
        dc.get_bbox_image(bbox)
        controller.buttons[snap_key].unobserve()
        controller.buttons[save_key].observe(save_bbox, names='value')
        controller.buttons[redo_key].observe(redo, names='value')

def save_bbox(change):
    global count_xy, bbox, snap_key, save_key, redo_key
    # save to disk
    x1 = bbox[0][0] * cam_width / 2 + cam_width / 2
    y1 = bbox[0][1] * cam_height / 2 + cam_height / 2
    x2 = bbox[1][0] * cam_width / 2 + cam_width / 2
    y2 = bbox[1][1] * cam_height / 2 + cam_height / 2
    uuid = f'bbox_{x1}_{y1}_{x2}_{y2}_{uuid1()}'
    image_path = os.path.join(DATASET_DIR, uuid + '.jpg')
    dc.save_image(image_path)
    count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))
    # fetch next bbox
    bbox = []
    count_xy = 2
    controller.buttons[save_key].unobserve()
    controller.buttons[redo_key].unobserve()

def redo(change):
    global count_xy, bbox, snap_key, save_key, redo_key
    count_xy = 2
    bbox = []
    controller.buttons[save_key].unobserve()
    controller.buttons[redo_key].unobserve()


### Connect Gamepad Controller
The cell below is to check the controller is well-connected by observing controller.connected property to prevent the controller is failed to initialized.

If the controller is observed well-connected, the selected axes and button will be assigned and linked for x, y sliders and save_snapshot button for data collection and store.


In [ ]:

def w_connected(change):
    global count_xy, bbox, snap_key, save_key, redo_key
    if change['new']==True:
        # dc.camera.observe(update_widget_image, 'value')

        # save_snapshot is workable once the controller buton for the save_snapshot is initialized.
        controller.buttons[snap_key].observe(snapshot_bbox, names='value')
        # controller.buttons[save_key].observe(save_bbox, names='value')
        # controller.buttons[redo_key].observe(redo, names='value')
        print(f'bbox point selection key connect to {snap_key}\n'
              f'save bbox key connected to {save_key} \n'
              f'redo bbox snap key connected to {redo_key} \n')

        axes_x = 0
        axes_y = 1
        widgets.jsdlink((controller.axes[axes_x], 'value'), (x_slider, 'value'))
        widgets.jsdlink((controller.axes[axes_y], 'value'), (y_slider, 'value'))
        x_slider.observe(update_xy_image, 'value')
        y_slider.observe(update_xy_image, 'value')
        print('x_slider connected to axes[%i] \ny_slider connected to axes[%i] \n' % (axes_x, axes_y))

controller.observe(w_connected, names='connected')

In [ ]:
display(widgets.HBox([widgets.VBox([target_widget, count_widget]), controller]))

Again, let's close the camera conneciton properly so that we can use the camera in other notebooks.

In [ ]:
dc.camera.stop()

### Next

Once you've collected enough data, we'll need to copy that data to our GPU desktop or cloud machine for training. First, we can call the following terminal command to compress our dataset folder into a single zip file.  

> If you're training on the JetBot itself, you can skip this step!

The ! prefix indicates that we want to run the cell as a shell (or terminal) command.

The -r flag in the zip command below indicates recursive so that we include all nested files, the -q flag indicates quiet so that the zip command doesn't print any output

In [ ]:
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q Jetbot_bbox_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}

You should see a file named road_following_<Date&Time>.zip in the Jupyter Lab file browser. You should download the zip file using the Jupyter Lab file browser by right clicking and selecting Download.